In [1]:
from papery.core.llm.core import get_llm
from pydantic import BaseModel, ConfigDict

In [3]:
# Parser prompt

parser_prompt = """
You will be provided with a user query. It is your job to return JSON text about the query in the specified format:
{
Key: "relevance"
{
Key: "output", bool
Details: A boolean stating whether the user's query is feasibly relevant to an academic research question.
Returns: true if the query is relevant, false if it's irrelevant
Examples: 
"What is 12*7" returns false because maths questions are not research related.
"Origins of COVID-19" returns true because this is a valid research question
---
Key: "reason", str
Details: Reason for your rating for `output`
}
Key: "query": str
Details: If relevance.output is false, return null, otherwise:
You should convert the user's query into a string which is appropriate for a vector search across the embeddings
of research papers. Therefore, the returned string for `query` should have the following properties:
- There should be minimal filler, the search string should be to the point and precise.
- You should not omit details from the user's query unless they aren't useful in a vector search.
- It should be concise, no longer than 25 words.
- The search query should not contain dates, authors or specific journals.
- The search query should not contain additional user requests, such as details on output format.
Example: "Tell me about ML for automated disease diagnosis in OCT retinal scans. Return it in an IEEE format. Search between 2010-2015"
Example output: "Computer Vision Machine Learning papers for disease diagnosis in Optical Coherence Tomography (OCT) retinal medical images"
Notes: This example output excludes filler, removes formatting references, removes dates, and clarifies acronyms to improve vector search appropriateness.
---
Key: "dates": tuple[int, int]
Details: If the user has NOT explicitly provided any dates in their query, return null for this key. If they have provided a date range,
provide the lower and upper bounds of their search range as a tuple.
}
"""


class Relevance(BaseModel):
    model_config = ConfigDict(strict=True)
    output: bool
    reason: str


class SearchPayload(BaseModel):
    model_config = ConfigDict(strict=True)
    relevance: Relevance
    query: str
    dates: tuple[int, int]


llm = get_llm("gpt-4o-mini", parser=SearchPayload)

In [4]:
await llm.call()